In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import os
os.chdir("../")

In [3]:
from ease_recommender import *
from npmi_recommender import *

import pickle as p

In [4]:
def create_mat(row, col, bool_to_int=True):
    # bool_to_int won't count duplicates in the same row, creates a different weighting basically
    if bool_to_int:
        data = np.ones_like(row, dtype=bool)
        return csr_matrix((data, (row, col))).astype(np.int64)
    else:
        data = np.ones_like(row, dtype=np.int64)
        return csr_matrix((data, (row, col)))

def check_if_all_terms_in_str(q, terms):
    for term in terms:
        if term not in q:
            return False

    return True

def get_cat2idx(category_type, D):
    if category_type == "track":
        return D["track2idx"]
    elif category_type == "album":
        return D["album2idx"]
    elif category_type == "artist":
        return D["artist2idx"]
    else:
        raise NotImplementedError

def find_match_using_terms(terms, cat2idx):
    matches = []
    for name in cat2idx.keys():
        if check_if_all_terms_in_str(name, terms):
            matches.append(name)

    if len(matches) > 1:
        raise Exception("Multiple matches found, filter down to a single match", matches)

    return matches[0]

In [5]:
print("loading cache data...")
D = p.load(open("cached_data/spotify_preprocessed.p", "rb"))

print("building csr matrices...")

# TODO: finish implementing track and album level recommendations

# track_mat = create_mat(D["playlist_indices"], D["track_indices"])
# album_mat = create_mat(D["playlist_indices"], D["album_indices"])
artist_mat = create_mat(D["playlist_indices"], D["artist_indices"])

print("done")

loading cache data...
building csr matrices...
done


In [6]:
cat2idx = get_cat2idx("artist", D)
idx2cat = {v:k for k, v in cat2idx.items()}

In [7]:
# use two items that you believe are similar to optimize the value of lambda_

a_name = find_match_using_terms(["sgeir", "7xUZ4069zcyBM4Bn10NQ1c"], cat2idx)
# a_name = find_match_using_terms(["7fNWySjsDn74LCawyJ27EQ"], cat2idx)

a = cat2idx[a_name]
print(f"{a_name=}")
print(f"Num Rows: {artist_mat[:, a].sum()}")

# a = cat2idx[find_match_using_terms(["Fleet Foxes"], cat2idx)]

# b = cat2idx[find_match_using_terms(["Fleet Foxes"], cat2idx)]
# b = cat2idx[find_match_using_terms(["Bon Iver", "4LEiUm1SRbFMgfqnQTwUbQ"], cat2idx)]
# b = cat2idx[find_match_using_terms(["SOHN"], cat2idx)]

# b_name = find_match_using_terms(["7fNWySjsDn74LCawyJ27EQ"], cat2idx)
b_name = find_match_using_terms(["Highas"], cat2idx)
b = cat2idx[b_name]
print(f"{b_name=}")
print(f"Num Rows: {artist_mat[:, b].sum()}")

a_name='Ásgeir (spotify:artist:7xUZ4069zcyBM4Bn10NQ1c)'
Num Rows: 1981
b_name='Highasakite (spotify:artist:5awQWdBpLqN2KFVRN8w56T)'
Num Rows: 412


In [8]:
c_name = find_match_using_terms(["Gabrielle (spotify:artist:4OovmAu23KrDlDQI2UbneL)"], cat2idx)
c = cat2idx[c_name]
print(f"{c_name=}")
print(f"Num Rows: {artist_mat[:, c].sum()}")

c_name='Gabrielle (spotify:artist:4OovmAu23KrDlDQI2UbneL)'
Num Rows: 71


In [10]:
mat = artist_mat
mat = csr_array(mat)

In [11]:
mat.shape

(1000000, 295860)

In [12]:
X = mat.T @ mat

In [13]:
X.shape

(295860, 295860)

In [14]:
import numpy as np
from scipy import sparse

def sparse_laplace_sppmi(X, alpha=1.0, normalize=False, zero_diag=True, non_neg=False):
    # Ensure input is CSR for fast row operations
    if not sparse.isspmatrix_csr(X):
        X = X.tocsr()
        
    # 1. Get Geometry of the Data
    # V: Vocabulary size (rows/cols)
    rows, cols = X.shape
    
    # N_raw: Total real observations
    N_raw = X.sum()
    
    # 2. Calculate "Smoothed" Marginals (The Global Statistics)
    # We pretend we added alpha to every cell, but we compute the sums analytically.
    
    # Virtual Total N = Real N + (alpha * Total Possible Cells)
    N_smoothed = N_raw + (alpha * rows * cols)
    
    # Raw marginals (sum of rows/cols)
    row_sums_raw = np.array(X.sum(axis=1)).flatten()
    col_sums_raw = np.array(X.sum(axis=0)).flatten()
    
    # Smoothed marginals = Raw Sum + (alpha * row_length)
    # Each row has 'cols' number of cells, so we add alpha * cols to the row sum
    P_x = (row_sums_raw + (alpha * cols)) / N_smoothed
    P_y = (col_sums_raw + (alpha * rows)) / N_smoothed
    
    # 3. Operate ONLY on Non-Zero Data (The Sparse Trick)
    # We extract the indices of existing data points to calculate their new PMI
    # efficiently, skipping the billions of zeros.
    
    # Create a copy to store results
    sppmi = X.copy().astype(np.float32)
    
    # Get indices of non-zero elements
    row_indices, col_indices = X.nonzero()
    
    # Get the raw counts
    raw_counts = np.array(X.data)
    
    # Smooth the counts: Count_new = Count_raw + alpha
    smoothed_counts = raw_counts + alpha
    
    # Calculate P(x,y) for these specific entries
    P_xy = smoothed_counts / N_smoothed
    
    # 4. Vectorized PMI Calculation
    # PMI = log( P(x,y) / (P(x) * P(y)) )
    # Note: P_x[row_indices] grabs the specific P(x) for every non-zero entry
    
    # Numerator is P_xy
    # Denominator is P(x) * P(y)
    denominator = P_x[row_indices] * P_y[col_indices]
    
    # Calculate PMI (using log2)
    pmi_values = np.log2(P_xy / denominator)
    
    if normalize:
        pmi_values = pmi_values / -np.log2(P_xy)
    
    # 7. Update the matrix data
    sppmi.data = pmi_values
    
    if non_neg:
        sppmi.data = np.where(sppmi.data > 0, sppmi.data, 0)
    
    if zero_diag:
        sppmi.setdiag(np.zeros(sppmi.shape[0]))
    
    # 8. Clean up (Remove explicit zeros to keep matrix sparse)
    sppmi.eliminate_zeros()
    
    return sppmi

alpha = .1 * .85

sppmi = sparse_laplace_sppmi(X, alpha=alpha, non_neg=True)

metric = (np.argsort(-sppmi[:, a].toarray()).tolist().index(b) + np.argsort(-sppmi[:, b].toarray()).tolist().index(a))/2
metric

In [20]:
import numpy as np
from scipy import sparse
from scipy.optimize import minimize_scalar

class SppmiOptimizerCache:
    """Caches global matrix invariants in memory for rapid SPPMI lookups."""
    def __init__(self, X):
        # CSC format is optimized for fast column slicing
        self.X_csc = X.tocsc() if not sparse.isspmatrix_csc(X) else X
        self.rows, self.cols = self.X_csc.shape
        
        # Precompute global statistics once
        self.N_raw = self.X_csc.sum()
        self.row_sums_raw = np.array(self.X_csc.sum(axis=1)).flatten()
        self.col_sums_raw = np.array(self.X_csc.sum(axis=0)).flatten()

In [21]:
def optimize_alpha_for_pair(cache, a, b, normalize=False, zero_diag=True, non_neg=False, bounds=(1e-4, 2.0)):
    # FIX: Slice as a 2D column block (a:a+1) to preserve CSC format and attributes
    col_a = cache.X_csc[:, a:a+1]
    rows_a, data_a = col_a.indices, col_a.data.astype(np.float32)
    
    col_b = cache.X_csc[:, b:b+1]
    rows_b, data_b = col_b.indices, col_b.data.astype(np.float32)
    
    # Pull constants out of the loop for speed
    rows, cols = cache.rows, cache.cols
    N_raw = cache.N_raw
    row_sums = cache.row_sums_raw
    col_sum_a = cache.col_sums_raw[a]
    col_sum_b = cache.col_sums_raw[b]

    def objective(alpha):
        N_smoothed = N_raw + (alpha * rows * cols)
        
        # --- Compute Column A ---
        P_x_a = (row_sums + (alpha * cols)) / N_smoothed
        P_y_a = (col_sum_a + (alpha * rows)) / N_smoothed
        P_xy_a = (data_a + alpha) / N_smoothed
        
        pmi_a = np.log2(P_xy_a / (P_x_a[rows_a] * P_y_a))
        if normalize:
            pmi_a /= -np.log2(P_xy_a)
        if non_neg:
            pmi_a = np.maximum(pmi_a, 0)
            
        dense_a = np.zeros(rows, dtype=np.float32)
        dense_a[rows_a] = pmi_a
        if zero_diag:
            dense_a[a] = 0
            
        # --- Compute Column B ---
        P_x_b = (row_sums + (alpha * cols)) / N_smoothed
        P_y_b = (col_sum_b + (alpha * rows)) / N_smoothed
        P_xy_b = (data_b + alpha) / N_smoothed
        
        pmi_b = np.log2(P_xy_b / (P_x_b[rows_b] * P_y_b))
        if normalize:
            pmi_b /= -np.log2(P_xy_b)
        if non_neg:
            pmi_b = np.maximum(pmi_b, 0)
            
        dense_b = np.zeros(rows, dtype=np.float32)
        dense_b[rows_b] = pmi_b
        if zero_diag:
            dense_b[b] = 0

        # --- Fast 1D Rank Evaluation ---
        rank_b_in_a = np.flatnonzero(np.argsort(-dense_a) == b)[0]
        rank_a_in_b = np.flatnonzero(np.argsort(-dense_b) == a)[0]
        
        return (rank_b_in_a + rank_a_in_b) / 2.0

    # Run scalar minimization using the bounded Brent method
    result = minimize_scalar(objective, bounds=bounds, method='bounded')
    return result

In [ ]:
# 1. Initialize the cache memory (Do this once)
cache = SppmiOptimizerCache(X)

In [25]:
# 3. Optimize alpha instantly
# res = optimize_alpha_for_pair(cache, a, b, non_neg=True, bounds=(0.001, 1.5))
res = optimize_alpha_for_pair(cache, a, b, non_neg=True, bounds=(0, 100))

print(f"Optimal Alpha: {res.x:.6f}")
print(f"Best Avg Rank: {res.fun}")

Optimal Alpha: 0.086675
Best Avg Rank: 10.0


In [72]:
import numpy as np
from scipy import sparse
from scipy.optimize import minimize_scalar

def assert_range(x, min=None, max=None):
    if min is not None:
        assert x.min() > min or np.isclose(x.min(), min), f"Min violation: {x.min()}"
    if max is not None:
        assert x.max() < max or np.isclose(x.max(), max), f"Max violation: {x.max()}"
    return x

def nan_to_const(x, const):
    return np.where(~np.isfinite(x), const, x)

class SppmiOptimizerCache:
    """Stores global matrix invariants in memory for rapid lookups."""
    def __init__(self, X):
        self.X_csc = X.tocsc() if not sparse.isspmatrix_csc(X) else X
        self.rows, self.cols = self.X_csc.shape
        self.N_raw = self.X_csc.sum()
        self.row_sums_raw = np.array(self.X_csc.sum(axis=1)).flatten()
        self.col_sums_raw = np.array(self.X_csc.sum(axis=0)).flatten()


def optimize_alpha_flexible(cache, a, b, ranking_type, mode="direct", zero_diag=True, bounds=(1e-4, 2.0)):
    X = cache.X_csc
    rows, cols = cache.rows, cache.cols
    N_raw = cache.N_raw
    row_sums_raw = cache.row_sums_raw
    col_sums_raw = cache.col_sums_raw

    # 1. Pre-extract sparse structural vectors for both target columns
    col_a = X[:, a:a+1]
    rows_a, data_a = col_a.indices, col_a.data.astype(np.float32)
    col_b = X[:, b:b+1]
    rows_b, data_b = col_b.indices, col_b.data.astype(np.float32)

    # 2. Precompute raw column dot-products if executing profile mode
    if mode == "dot_product":
        raw_dots_a = (X[:, a].T @ X).toarray().flatten()
        raw_dots_b = (X[:, b].T @ X).toarray().flatten()

    def get_ranking_score(idx, alpha):
        N_smoothed = N_raw + (alpha * rows * cols)
        curr_rows, curr_data = (rows_a, data_a) if idx == a else (rows_b, data_b)
        
        if mode == "direct":
            # --- MODE: DIRECT (Matches your original high-performing version) ---
            P_x = (row_sums_raw + (alpha * cols)) / N_smoothed
            P_y_idx = (col_sums_raw[idx] + (alpha * rows)) / N_smoothed
            
            # Construct dense joint probability vector for column 'idx'
            P_xy = np.full(rows, alpha / N_smoothed, dtype=np.float32)
            P_xy[curr_rows] = (curr_data + alpha) / N_smoothed
            
            px_given_y = P_xy / P_y_idx
            lift = P_xy / (P_x * P_y_idx)
            lift_normalizer = np.minimum(1.0 / P_x, 1.0 / P_y_idx)
            px_metric = P_x
            pxy_normalization_term = P_xy
            
        else:
            # --- MODE: DOT PRODUCT (Matches your second snippet's formulas) ---
            px_metric = (col_sums_raw + (alpha * rows)) / N_smoothed
            curr_dots = raw_dots_a if idx == a else raw_dots_b
            
            num_counts = curr_dots + (alpha * col_sums_raw[idx]) + (alpha * col_sums_raw) + (rows * (alpha ** 2))
            col_sum_idx_smoothed = col_sums_raw[idx] + (alpha * rows)
            
            px_given_y = num_counts / (N_smoothed * col_sum_idx_smoothed)
            lift = num_counts / (col_sum_idx_smoothed * (col_sums_raw + (alpha * rows)))
            lift_normalizer = np.minimum(1.0 / px_metric, 1.0 / px_metric[idx])
            
            # Fixed NPMI mapping for profile mode
            P_xy_cell = np.full(rows, alpha / N_smoothed, dtype=np.float32)
            P_xy_cell[curr_rows] = (curr_data + alpha) / N_smoothed
            pxy_normalization_term = P_xy_cell

        # --- Apply Metric Type ---
        if ranking_type == "px":
            score = assert_range(px_metric, 0, 1)
        elif ranking_type == "1/px":
            score = assert_range(1.0 / px_metric, 0, None)
        elif ranking_type == "px_given_y":
            score = assert_range(px_given_y, 0, 1)
        elif ranking_type == "npmi":
            pmi = np.log2(lift) / -np.log2(pxy_normalization_term)
            score = assert_range(nan_to_const(pmi, -1.0), -1, 1)
        elif ranking_type == "npmi_alt":
            score = assert_range(np.log2(lift) / np.log2(lift_normalizer), None, 1)
        elif ranking_type == "lift":
            score = assert_range(lift, 0, None)
        elif ranking_type == "weighted_lift":
            score = assert_range(lift * px_given_y, 0, None)
        elif ranking_type == "normalized_lift":
            score = assert_range(lift / lift_normalizer, 0, 1)
        elif ranking_type == "normalized_weighted_lift":
            score = assert_range((lift / lift_normalizer) * px_given_y, 0, 1)
        else:
            raise NotImplementedError(f"Unknown ranking type: {ranking_type}")
            
        if zero_diag:
            score[idx] = 0
            
        return score

    def objective(alpha):
        scores_a = get_ranking_score(a, alpha)
        scores_b = get_ranking_score(b, alpha)

        rank_b_in_a = np.flatnonzero(np.argsort(-scores_a) == b)[0]
        rank_a_in_b = np.flatnonzero(np.argsort(-scores_b) == a)[0]
        
        return (rank_b_in_a + rank_a_in_b) / 2.0

    result = minimize_scalar(objective, bounds=bounds, method='bounded')
    return result

In [73]:
# 1. Initialize the global data cache once
cache = SppmiOptimizerCache(X)

In [74]:
# metric_to_test = "px"

# # 3. Optimize alpha instantly
# res = optimize_alpha_flexible(cache, a, b, ranking_type=metric_to_test, bounds=(0, 100.0))

# print(f"Metrics Tested: {metric_to_test}")
# print(f"Optimal Alpha:  {res.x:.6f}")
# print(f"Best Avg Rank:  {res.fun}")

In [75]:
# metric_to_test = "1/px"

# # 3. Optimize alpha instantly
# res = optimize_alpha_flexible(cache, a, b, ranking_type=metric_to_test, bounds=(0, 100.0))

# print(f"Metrics Tested: {metric_to_test}")
# print(f"Optimal Alpha:  {res.x:.6f}")
# print(f"Best Avg Rank:  {res.fun}")

In [76]:
# metric_to_test = "px_given_y"

# # 3. Optimize alpha instantly
# res = optimize_alpha_flexible(cache, a, b, ranking_type=metric_to_test, bounds=(0, 100.0))

# print(f"Metrics Tested: {metric_to_test}")
# print(f"Optimal Alpha:  {res.x:.6f}")
# print(f"Best Avg Rank:  {res.fun}")

In [77]:
metric_to_test = "npmi"

# 3. Optimize alpha instantly
res = optimize_alpha_flexible(cache, a, b, ranking_type=metric_to_test, bounds=(0, 100.0))

print(f"Metrics Tested: {metric_to_test}")
print(f"Optimal Alpha:  {res.x:.6f}")
print(f"Best Avg Rank:  {res.fun}")

metric_to_test = "npmi_alt"

# 3. Optimize alpha instantly
res = optimize_alpha_flexible(cache, a, b, ranking_type=metric_to_test, bounds=(0, 100.0))

print(f"Metrics Tested: {metric_to_test}")
print(f"Optimal Alpha:  {res.x:.6f}")
print(f"Best Avg Rank:  {res.fun}")

metric_to_test = "lift"

# 3. Optimize alpha instantly
res = optimize_alpha_flexible(cache, a, b, ranking_type=metric_to_test, bounds=(0, 100.0))

print(f"Metrics Tested: {metric_to_test}")
print(f"Optimal Alpha:  {res.x:.6f}")
print(f"Best Avg Rank:  {res.fun}")

Metrics Tested: npmi
Optimal Alpha:  0.039425
Best Avg Rank:  13.0
Metrics Tested: npmi_alt
Optimal Alpha:  0.051934
Best Avg Rank:  9.5
Metrics Tested: lift
Optimal Alpha:  0.086675
Best Avg Rank:  10.0


In [37]:
metric_to_test = "weighted_lift"

# 3. Optimize alpha instantly
res = optimize_alpha_flexible(cache, a, b, ranking_type=metric_to_test, bounds=(0, 100.0))

print(f"Metrics Tested: {metric_to_test}")
print(f"Optimal Alpha:  {res.x:.6f}")
print(f"Best Avg Rank:  {res.fun}")

Metrics Tested: weighted_lift
Optimal Alpha:  0.008972
Best Avg Rank:  77.0


In [36]:
metric_to_test = "normalized_lift"

# 3. Optimize alpha instantly
res = optimize_alpha_flexible(cache, a, b, ranking_type=metric_to_test, bounds=(0, 100.0))

print(f"Metrics Tested: {metric_to_test}")
print(f"Optimal Alpha:  {res.x:.6f}")
print(f"Best Avg Rank:  {res.fun}")

Metrics Tested: normalized_lift
Optimal Alpha:  58.373215
Best Avg Rank:  481.5


In [35]:
metric_to_test = "normalized_weighted_lift"

# 3. Optimize alpha instantly
res = optimize_alpha_flexible(cache, a, b, ranking_type=metric_to_test, bounds=(0, 100.0))

print(f"Metrics Tested: {metric_to_test}")
print(f"Optimal Alpha:  {res.x:.6f}")
print(f"Best Avg Rank:  {res.fun}")

Metrics Tested: normalized_weighted_lift
Optimal Alpha:  58.373215
Best Avg Rank:  481.5


In [48]:
import numpy as np
from scipy import sparse
from scipy.optimize import minimize_scalar

def optimize_alpha_fixed(cache, a, b, ranking_type, mode="direct", zero_diag=True, bounds=(1e-4, 2.0)):
    X = cache.X_csc
    rows, cols = cache.rows, cache.cols
    N_raw = cache.N_raw
    row_sums_raw = cache.row_sums_raw
    col_sums_raw = cache.col_sums_raw

    col_a = X[:, a:a+1]
    rows_a, data_a = col_a.indices, col_a.data.astype(np.float32)
    col_b = X[:, b:b+1]
    rows_b, data_b = col_b.indices, col_b.data.astype(np.float32)

    if mode == "dot_product":
        raw_dots_a = (X[:, a].T @ X).toarray().flatten()
        raw_dots_b = (X[:, b].T @ X).toarray().flatten()

    def get_ranking_score(idx, alpha):
        N_smoothed = N_raw + (alpha * rows * cols)
        curr_rows, curr_data = (rows_a, data_a) if idx == a else (rows_b, data_b)
        
        # Track EXACTLY which rows have real, raw co-occurrences
        is_nonzero = np.zeros(rows, dtype=bool)
        
        if mode == "direct":
            P_x = (row_sums_raw + (alpha * cols)) / N_smoothed
            P_y_idx = (col_sums_raw[idx] + (alpha * rows)) / N_smoothed
            
            P_xy = np.full(rows, alpha / N_smoothed, dtype=np.float32)
            P_xy[curr_rows] = (curr_data + alpha) / N_smoothed
            
            px_given_y = P_xy / P_y_idx
            lift = P_xy / (P_x * P_y_idx)
            lift_normalizer = np.minimum(1.0 / P_x, 1.0 / P_y_idx)
            px_metric = P_x
            pxy_normalization_term = P_xy
            
            # Mark raw hits
            is_nonzero[curr_rows] = True
            
        else: # dot_product mode
            px_metric = (col_sums_raw + (alpha * rows)) / N_smoothed
            curr_dots = raw_dots_a if idx == a else raw_dots_b
            
            num_counts = curr_dots + (alpha * col_sums_raw[idx]) + (alpha * col_sums_raw) + (rows * (alpha ** 2))
            col_sum_idx_smoothed = col_sums_raw[idx] + (alpha * rows)
            
            px_given_y = num_counts / (N_smoothed * col_sum_idx_smoothed)
            lift = num_counts / (col_sum_idx_smoothed * (col_sums_raw + (alpha * rows)))
            lift_normalizer = np.minimum(1.0 / px_metric, 1.0 / px_metric[idx])
            
            P_xy_cell = np.full(rows, alpha / N_smoothed, dtype=np.float32)
            P_xy_cell[curr_rows] = (curr_data + alpha) / N_smoothed
            pxy_normalization_term = P_xy_cell
            
            # In profile mode, a "hit" means they share at least one historical neighbor
            is_nonzero[curr_dots > 0] = True

        # --- Compute Target Metrics ---
        if ranking_type == "px":
            score = px_metric
        elif ranking_type == "1/px":
            score = 1.0 / px_metric
        elif ranking_type == "px_given_y":
            score = px_given_y
        elif ranking_type == "npmi":
            pmi = np.log2(lift) / -np.log2(pxy_normalization_term)
            score = nan_to_const(pmi, -1.0)
        elif ranking_type == "npmi_alt":
            score = np.log2(lift) / np.log2(lift_normalizer)
        elif ranking_type == "lift":
            score = lift
        elif ranking_type == "weighted_lift":
            score = lift * px_given_y
        elif ranking_type == "normalized_lift":
            score = lift / lift_normalizer
        elif ranking_type == "normalized_weighted_lift":
            score = (lift / lift_normalizer) * px_given_y
        else:
            raise NotImplementedError
            
        # FIX: Force structural zeros to absolute zero so they cannot outrank real matches
        score[~is_nonzero] = 0.0
            
        if zero_diag:
            score[idx] = 0
            
        return score

    def objective(alpha):
        scores_a = get_ranking_score(a, alpha)
        scores_b = get_ranking_score(b, alpha)

        rank_b_in_a = np.flatnonzero(np.argsort(-scores_a) == b)[0]
        rank_a_in_b = np.flatnonzero(np.argsort(-scores_b) == a)[0]
        
        return (rank_b_in_a + rank_a_in_b) / 2.0

    result = minimize_scalar(objective, bounds=bounds, method='bounded')
    return result

In [80]:
# 1. Initialize the global data cache once
cache = SppmiOptimizerCache(X)

In [66]:
metric_to_test = "weighted_lift"

# 3. Optimize alpha instantly
res = optimize_alpha_fixed(cache, a, b, ranking_type=metric_to_test, bounds=(0, 1e1))

print(f"Metrics Tested: {metric_to_test}")
print(f"Optimal Alpha:  {res.x:.6f}")
print(f"Best Avg Rank:  {res.fun}")

Metrics Tested: weighted_lift
Optimal Alpha:  0.003879
Best Avg Rank:  74.0


In [65]:
metric_to_test = "normalized_lift"

# 3. Optimize alpha instantly
res = optimize_alpha_fixed(cache, a, b, ranking_type=metric_to_test, bounds=(0, 1e1))

print(f"Metrics Tested: {metric_to_test}")
print(f"Optimal Alpha:  {res.x:.6f}")
print(f"Best Avg Rank:  {res.fun}")

Metrics Tested: normalized_lift
Optimal Alpha:  0.030427
Best Avg Rank:  138.0


In [64]:
metric_to_test = "normalized_weighted_lift"

# 3. Optimize alpha instantly
res = optimize_alpha_fixed(cache, a, b, ranking_type=metric_to_test, bounds=(0, 1e1))

print(f"Metrics Tested: {metric_to_test}")
print(f"Optimal Alpha:  {res.x:.6f}")
print(f"Best Avg Rank:  {res.fun}")

Metrics Tested: normalized_weighted_lift
Optimal Alpha:  0.002576
Best Avg Rank:  238.5


In [67]:
import numpy as np
from scipy import sparse
from scipy.optimize import minimize_scalar

def optimize_alpha_regularized(cache, a, b, ranking_type, mode="direct", zero_diag=True, tau=2.0, bounds=(1e-4, 2.0)):
    X = cache.X_csc
    rows, cols = cache.rows, cache.cols
    N_raw = cache.N_raw
    row_sums_raw = cache.row_sums_raw
    col_sums_raw = cache.col_sums_raw

    col_a = X[:, a:a+1]
    rows_a, data_a = col_a.indices, col_a.data.astype(np.float32)
    col_b = X[:, b:b+1]
    rows_b, data_b = col_b.indices, col_b.data.astype(np.float32)

    if mode == "dot_product":
        raw_dots_a = (X[:, a].T @ X).toarray().flatten()
        raw_dots_b = (X[:, b].T @ X).toarray().flatten()

    def get_ranking_score(idx, alpha):
        N_smoothed = N_raw + (alpha * rows * cols)
        curr_rows, curr_data = (rows_a, data_a) if idx == a else (rows_b, data_b)
        
        # 1. Create a raw co-occurrence count vector to calculate the shrinkage modifier
        raw_counts = np.zeros(rows, dtype=np.float32)
        
        if mode == "direct":
            P_x = (row_sums_raw + (alpha * cols)) / N_smoothed
            P_y_idx = (col_sums_raw[idx] + (alpha * rows)) / N_smoothed
            
            P_xy = np.full(rows, alpha / N_smoothed, dtype=np.float32)
            P_xy[curr_rows] = (curr_data + alpha) / N_smoothed
            
            px_given_y = P_xy / P_y_idx
            lift = P_xy / (P_x * P_y_idx)
            lift_normalizer = np.minimum(1.0 / P_x, 1.0 / P_y_idx)
            px_metric = P_x
            pxy_normalization_term = P_xy
            
            raw_counts[curr_rows] = curr_data
            
        else: # dot_product mode
            px_metric = (col_sums_raw + (alpha * rows)) / N_smoothed
            curr_dots = raw_dots_a if idx == a else raw_dots_b
            
            num_counts = curr_dots + (alpha * col_sums_raw[idx]) + (alpha * col_sums_raw) + (rows * (alpha ** 2))
            col_sum_idx_smoothed = col_sums_raw[idx] + (alpha * rows)
            
            px_given_y = num_counts / (N_smoothed * col_sum_idx_smoothed)
            lift = num_counts / (col_sum_idx_smoothed * (col_sums_raw + (alpha * rows)))
            lift_normalizer = np.minimum(1.0 / px_metric, 1.0 / px_metric[idx])
            
            P_xy_cell = np.full(rows, alpha / N_smoothed, dtype=np.float32)
            P_xy_cell[curr_rows] = (curr_data + alpha) / N_smoothed
            pxy_normalization_term = P_xy_cell
            
            raw_counts = curr_dots

        # 2. Build the continuous shrinkage modifier
        # If raw_count is 0, modifier is 0. If raw_count is large, modifier approaches 1.0.
        shrinkage_modifier = raw_counts / (raw_counts + tau)

        # 3. Calculate Core Metrics
        if ranking_type == "px":
            score = px_metric
        elif ranking_type == "1/px":
            score = 1.0 / px_metric
        elif ranking_type == "px_given_y":
            score = px_given_y * shrinkage_modifier
        elif ranking_type == "npmi":
            pmi = np.log2(lift) / -np.log2(pxy_normalization_term)
            score = nan_to_const(pmi, -1.0) * shrinkage_modifier
        elif ranking_type == "npmi_alt":
            score = (np.log2(lift) / np.log2(lift_normalizer)) * shrinkage_modifier
        elif ranking_type == "lift":
            score = lift * shrinkage_modifier
        elif ranking_type == "weighted_lift":
            score = (lift * px_given_y) * shrinkage_modifier
        elif ranking_type == "normalized_lift":
            score = (lift / lift_normalizer) * shrinkage_modifier
        elif ranking_type == "normalized_weighted_lift":
            score = ((lift / lift_normalizer) * px_given_y) * shrinkage_modifier
        else:
            raise NotImplementedError
            
        if zero_diag:
            score[idx] = 0
            
        return score

    def objective(alpha):
        scores_a = get_ranking_score(a, alpha)
        scores_b = get_ranking_score(b, alpha)

        rank_b_in_a = np.flatnonzero(np.argsort(-scores_a) == b)[0]
        rank_a_in_b = np.flatnonzero(np.argsort(-scores_b) == a)[0]
        
        return (rank_b_in_a + rank_a_in_b) / 2.0

    result = minimize_scalar(objective, bounds=bounds, method='bounded')
    return result

In [71]:
metric_to_test = "weighted_lift"

# 3. Optimize alpha instantly
res = optimize_alpha_regularized(cache, a, b, ranking_type=metric_to_test, bounds=(0, 1e1))

print(f"Metrics Tested: {metric_to_test}")
print(f"Optimal Alpha:  {res.x:.6f}")
print(f"Best Avg Rank:  {res.fun}")

metric_to_test = "normalized_lift"

# 3. Optimize alpha instantly
res = optimize_alpha_regularized(cache, a, b, ranking_type=metric_to_test, bounds=(0, 1e1))

print(f"Metrics Tested: {metric_to_test}")
print(f"Optimal Alpha:  {res.x:.6f}")
print(f"Best Avg Rank:  {res.fun}")

metric_to_test = "normalized_weighted_lift"

# 3. Optimize alpha instantly
res = optimize_alpha_regularized(cache, a, b, ranking_type=metric_to_test, bounds=(0, 1e1))

print(f"Metrics Tested: {metric_to_test}")
print(f"Optimal Alpha:  {res.x:.6f}")
print(f"Best Avg Rank:  {res.fun}")

Metrics Tested: weighted_lift
Optimal Alpha:  0.004535
Best Avg Rank:  78.5
Metrics Tested: normalized_lift
Optimal Alpha:  0.031134
Best Avg Rank:  139.0
Metrics Tested: normalized_weighted_lift
Optimal Alpha:  0.007992
Best Avg Rank:  242.0


In [81]:
import numpy as np
from scipy import sparse
from scipy.optimize import minimize_scalar

def nan_to_const(x, const):
    return np.where(~np.isfinite(x), const, x)

class SppmiOptimizerCache:
    def __init__(self, X):
        self.X_csc = X.tocsc() if not sparse.isspmatrix_csc(X) else X
        self.rows, self.cols = self.X_csc.shape
        self.N_raw = self.X_csc.sum()
        self.row_sums_raw = np.array(self.X_csc.sum(axis=1)).flatten()
        self.col_sums_raw = np.array(self.X_csc.sum(axis=0)).flatten()

def optimize_alpha_advanced(cache, a, b, ranking_type, mode="direct", 
                            symmetrize=False, gamma=1.0, tau=2.0, 
                            zero_diag=True, bounds=(1e-4, 2.0)):
    X = cache.X_csc
    rows, cols = cache.rows, cache.cols
    N_raw = cache.N_raw
    row_sums_raw = cache.row_sums_raw
    col_sums_raw = cache.col_sums_raw

    col_a = X[:, a:a+1]
    rows_a, data_a = col_a.indices, col_a.data.astype(np.float32)
    col_b = X[:, b:b+1]
    rows_b, data_b = col_b.indices, col_b.data.astype(np.float32)

    if mode == "dot_product":
        raw_dots_a = (X[:, a].T @ X).toarray().flatten()
        raw_dots_b = (X[:, b].T @ X).toarray().flatten()

    def get_ranking_score(idx, alpha):
        N_smoothed = N_raw + (alpha * rows * cols)
        curr_rows, curr_data = (rows_a, data_a) if idx == a else (rows_b, data_b)
        raw_counts = np.zeros(rows, dtype=np.float32)
        
        if mode == "direct":
            P_x = (row_sums_raw + (alpha * cols)) / N_smoothed
            P_y_idx = (col_sums_raw[idx] + (alpha * rows)) / N_smoothed
            
            P_xy = np.full(rows, alpha / N_smoothed, dtype=np.float32)
            P_xy[curr_rows] = (curr_data + alpha) / N_smoothed
            
            lift = P_xy / (P_x * P_y_idx)
            px_given_y = P_xy / P_y_idx  # Forward: P(x|y)
            py_given_x = P_xy / P_x      # Backward: P(y|x)
            lift_normalizer = np.minimum(1.0 / P_x, 1.0 / P_y_idx)
            pxy_normalization_term = P_xy
            raw_counts[curr_rows] = curr_data
        else:
            # Profile mode
            P_x = (col_sums_raw + (alpha * rows)) / N_smoothed
            P_y_idx = (col_sums_raw[idx] + (alpha * rows)) / N_smoothed
            curr_dots = raw_dots_a if idx == a else raw_dots_b
            num_counts = curr_dots + (alpha * col_sums_raw[idx]) + (alpha * col_sums_raw) + (rows * (alpha ** 2))
            
            lift = num_counts / ((col_sums_raw[idx] + (alpha * rows)) * (col_sums_raw + (alpha * rows)))
            px_given_y = num_counts / (N_smoothed * (col_sums_raw[idx] + (alpha * rows)))
            py_given_x = num_counts / (N_smoothed * (col_sums_raw + (alpha * rows)))
            lift_normalizer = np.minimum(1.0 / P_x, 1.0 / P_y_idx)
            pxy_normalization_term = num_counts / (N_smoothed ** 2)
            raw_counts = curr_dots

        # Apply Strategy 2: Parameterized Exponent Power Lift
        # Generalizes lift/pmi by adjusting the denominator exponent control
        soft_power_lift = P_xy / ((P_x ** gamma) * (P_y_idx ** gamma)) if mode == "direct" else lift

        # Apply Strategy 1: Explicit Symmetrization via Geometric Mean
        if ranking_type == "weighted_lift":
            forward = lift * px_given_y
            backward = lift * py_given_x
            score = np.sqrt(forward * backward) if symmetrize else forward
            
        elif ranking_type == "normalized_lift":
            # Modifying normalized lift's denominator behavior with gamma
            tuned_normalizer = np.minimum(1.0 / (P_x ** gamma), 1.0 / (P_y_idx ** gamma))
            score = lift / tuned_normalizer
            
        elif ranking_type == "normalized_weighted_lift":
            forward = (lift / lift_normalizer) * px_given_y
            backward = (lift / lift_normalizer) * py_given_x
            score = np.sqrt(forward * backward) if symmetrize else forward
            
        elif ranking_type == "soft_power_lift":
            score = soft_power_lift
            
        elif ranking_type == "npmi_alt":
            score = np.log2(lift) / np.log2(lift_normalizer)
        else:
            score = lift

        # Apply count regularization shrink
        if tau > 0:
            score *= (raw_counts / (raw_counts + tau))
            
        if zero_diag:
            score[idx] = 0
            
        return score

    def objective(alpha):
        scores_a = get_ranking_score(a, alpha)
        scores_b = get_ranking_score(b, alpha)
        rank_b_in_a = np.flatnonzero(np.argsort(-scores_a) == b)[0]
        rank_a_in_b = np.flatnonzero(np.argsort(-scores_b) == a)[0]
        return (rank_b_in_a + rank_a_in_b) / 2.0

    result = minimize_scalar(objective, bounds=bounds, method='bounded')
    return result

In [83]:
metric_to_test = "weighted_lift"

# 3. Optimize alpha instantly
res = optimize_alpha_advanced(cache, a, b, ranking_type=metric_to_test, bounds=(0, 1e1))

print(f"Metrics Tested: {metric_to_test}")
print(f"Optimal Alpha:  {res.x:.6f}")
print(f"Best Avg Rank:  {res.fun}")

metric_to_test = "normalized_lift"

# 3. Optimize alpha instantly
res = optimize_alpha_advanced(cache, a, b, ranking_type=metric_to_test, bounds=(0, 1e1))

print(f"Metrics Tested: {metric_to_test}")
print(f"Optimal Alpha:  {res.x:.6f}")
print(f"Best Avg Rank:  {res.fun}")

metric_to_test = "normalized_weighted_lift"

# 3. Optimize alpha instantly
res = optimize_alpha_advanced(cache, a, b, ranking_type=metric_to_test, bounds=(0, 1e1))

print(f"Metrics Tested: {metric_to_test}")
print(f"Optimal Alpha:  {res.x:.6f}")
print(f"Best Avg Rank:  {res.fun}")

Metrics Tested: weighted_lift
Optimal Alpha:  0.004535
Best Avg Rank:  78.5
Metrics Tested: normalized_lift
Optimal Alpha:  0.031134
Best Avg Rank:  139.0
Metrics Tested: normalized_weighted_lift
Optimal Alpha:  0.007992
Best Avg Rank:  242.0


In [87]:
metric_to_test = "soft_power_lift"

# 3. Optimize alpha instantly
res = optimize_alpha_advanced(cache, a, b, ranking_type=metric_to_test, bounds=(0, 1e1))

print(f"Metrics Tested: {metric_to_test}")
print(f"Optimal Alpha:  {res.x:.6f}")
print(f"Best Avg Rank:  {res.fun}")

Metrics Tested: soft_power_lift
Optimal Alpha:  0.066653
Best Avg Rank:  8.5


In [105]:
metric_to_test = "soft_power_lift"

# 3. Optimize alpha instantly
res = optimize_alpha_advanced(cache, a, b, ranking_type=metric_to_test, bounds=(0, 1e1), gamma=1.3)

print(f"Metrics Tested: {metric_to_test}")
print(f"Optimal Alpha:  {res.x:.6f}")
print(f"Best Avg Rank:  {res.fun}")

Metrics Tested: soft_power_lift
Optimal Alpha:  0.112115
Best Avg Rank:  6.0


In [121]:
metric_to_test = "soft_power_lift"

# 3. Optimize alpha instantly
res = optimize_alpha_advanced(cache, a, b, ranking_type=metric_to_test, bounds=(0, 1e1), gamma=1.3, tau=4)

print(f"Metrics Tested: {metric_to_test}")
print(f"Optimal Alpha:  {res.x:.6f}")
print(f"Best Avg Rank:  {res.fun}")

Metrics Tested: soft_power_lift
Optimal Alpha:  0.096834
Best Avg Rank:  5.5


In [127]:
# metric_to_test = "normalized_weighted_lift"

# # 3. Optimize alpha instantly
# res = optimize_alpha_advanced(cache, a, b, ranking_type=metric_to_test, bounds=(0, 1e1), gamma=1.3, tau=4, symmetrize=True)

# print(f"Metrics Tested: {metric_to_test}")
# print(f"Optimal Alpha:  {res.x:.6f}")
# print(f"Best Avg Rank:  {res.fun}")

In [85]:
metric_to_test = "soft_power_lift"

# 3. Optimize alpha instantly
res = optimize_alpha_advanced(cache, a, b, ranking_type=metric_to_test, bounds=(0, 1e1))

print(f"Metrics Tested: {metric_to_test}")
print(f"Optimal Alpha:  {res.x:.6f}")
print(f"Best Avg Rank:  {res.fun}")

metric_to_test = "normalized_lift"

# 3. Optimize alpha instantly
res = optimize_alpha_advanced(cache, a, b, ranking_type=metric_to_test, bounds=(0, 1e1))

print(f"Metrics Tested: {metric_to_test}")
print(f"Optimal Alpha:  {res.x:.6f}")
print(f"Best Avg Rank:  {res.fun}")

metric_to_test = "normalized_weighted_lift"

# 3. Optimize alpha instantly
res = optimize_alpha_advanced(cache, a, b, ranking_type=metric_to_test, bounds=(0, 1e1))

print(f"Metrics Tested: {metric_to_test}")
print(f"Optimal Alpha:  {res.x:.6f}")
print(f"Best Avg Rank:  {res.fun}")

KeyboardInterrupt: 

In [ ]:
px = mat.mean(axis=0)
pxy = X / mat.shape[0]

In [94]:
def assert_range(x, min=None, max=None):
    if min is not None:
        assert x.min() > min or np.isclose(x.min(), min), x.min()
    
    if max is not None:
        assert x.max() < max or np.isclose(x.max(), max), x.max()
        
    return x

def nan_to_const(x, const):
    return np.where(~np.isfinite(x), const, x)

def generate_ranking_score(a, ranking_type):
    px_given_y = (mat[:, a:a+1] * mat).sum(axis=0) / mat[:, a].sum()
    
    lift = px_given_y / px
    
    lift_normalizer = np.minimum(1/px, 1/px[a])
    
    if ranking_type == "px":
        return assert_range(px, 0, 1)
    elif ranking_type == "1/px":
        return assert_range(1/px, 0, None)
    elif ranking_type == "px_given_y":
        return assert_range(px_given_y, 0, 1)
    #         return assert_range(np.log2(lift) / np.log2(np.maximum(1/px, 1/px[a])), None, 1)
    elif ranking_type == "npmi":
        return assert_range(nan_to_const(np.log2(lift) / -np.log2(pxy[a].toarray()), -1), -1, 1)
    elif ranking_type == "npmi_alt":
        return assert_range(np.log2(lift) / np.log2(lift_normalizer), None, 1)
#     elif ranking_type == "npmi_alt2":
    elif ranking_type == "lift":
        return assert_range(lift, 0, None)
    elif ranking_type == "weighted_lift":
        return assert_range(lift * px_given_y, 0, None)
    elif ranking_type == "normalized_lift":
        return assert_range(lift / lift_normalizer, 0, 1)
#     elif ranking_type == "normalized_lift2":
#         return assert_range(lift / np.maximum(1/px, 1/px[a]), 0, 1)
    elif ranking_type == "normalized_weighted_lift":
        return assert_range((lift / lift_normalizer) * px_given_y, 0, 1)
#         score = lift / lift_normalizer
#         print(score.min(), score.max())
#         score = assert_range(score * px_given_y, 0, 1)
#         print(score.min(), score.max())
#         return score
#     elif ranking_type == "normalized_weighted_lift2":
#         return assert_range((lift / np.maximum(1/px, 1/px[a])) * px_given_y, 0, 1)
#     elif ranking_type == "normalized_weighted_lift2":
#         return assert_range((lift / np.maximum(1/px, 1/px[a])) * px_given_y, 0, 1)
    else:
        raise NotImplementedError

In [95]:
from tqdm.auto import tqdm

for ranking_type in tqdm(["px", "1/px", "px_given_y", "npmi", "npmi_alt", "lift", "weighted_lift", "normalized_lift", "normalized_weighted_lift"]):
    metric = (np.argsort(-generate_ranking_score(a, ranking_type)).tolist().index(b) +
                np.argsort(-generate_ranking_score(b, ranking_type)).tolist().index(a))/2
    
    print(ranking_type, metric)

  0%|          | 0/9 [00:00<?, ?it/s]

px 5559.0
1/px 290301.0
px_given_y 495.0


C:\Users\johns\AppData\Local\Temp\ipykernel_7624\3468900810.py:28: RuntimeWarning: divide by zero encountered in log2
  return assert_range(nan_to_const(np.log2(lift) / -np.log2(pxy[a].toarray()), -1), -1, 1)
C:\Users\johns\AppData\Local\Temp\ipykernel_7624\3468900810.py:28: RuntimeWarning: invalid value encountered in divide
  return assert_range(nan_to_const(np.log2(lift) / -np.log2(pxy[a].toarray()), -1), -1, 1)


npmi 904.0


C:\Users\johns\AppData\Local\Temp\ipykernel_7624\3468900810.py:30: RuntimeWarning: divide by zero encountered in log2
  return assert_range(np.log2(lift) / np.log2(lift_normalizer), None, 1)


npmi_alt 1681.5
lift 2132.0
weighted_lift 127.5
normalized_lift 1601.0
normalized_weighted_lift 242.0


In [110]:
from scipy.optimize import minimize, minimize_scalar

In [118]:
def generate_ranking_score(a, p):
    px_given_y = (mat[:, a:a+1] * mat).sum(axis=0) / mat[:, a].sum()
    
    lift = px_given_y / px
    
    lift_normalizer = np.minimum(1/px, 1/px[a])
    
#     return (px_given_y**p) * (lift / lift_normalizer)**(2 - p)
    return lift**p / lift_normalizer**(2 - p)
    
def f(p):
    metric = (np.argsort(-generate_ranking_score(a, p)).tolist().index(b) +
                np.argsort(-generate_ranking_score(b, p)).tolist().index(a))/2
    
    print(metric)
    
    return metric

In [119]:
res = minimize_scalar(f, bounds=(0, 2))

1757.0
1668.0
1873.5
1606.5
1588.0
1617.5
1604.5
1589.0
1596.5
1602.0
1588.5
1588.5
1602.0
1588.5
1602.0
1588.0
1588.0
1588.0
1588.0
1588.0
1588.0


In [107]:
def generate_ranking_score(a, p):
    px_given_y = (mat[:, a:a+1] * mat).sum(axis=0) / mat[:, a].sum()
    
    lift = px_given_y / px
    
    lift_normalizer = 1/np.minimum(1/px, 1/px[a])
    
    return (px_given_y**p[0]) * (lift**p[1]) * (lift_normalizer**p[2])
    
def f(p):
    metric = (np.argsort(-generate_ranking_score(a, p)).tolist().index(b) +
                np.argsort(-generate_ranking_score(b, p)).tolist().index(a))/2
    
    print(metric)
    
    return metric

In [108]:
f(np.ones(3))

242.5


242.5

In [109]:
res = minimize(f, np.ones(3), method="Powell")

242.5
242.5
300.5


C:\Users\johns\AppData\Local\Temp\ipykernel_7624\3385663312.py:8: RuntimeWarning: divide by zero encountered in power
  return (px_given_y**p[0]) * (lift**p[1]) * (lift_normalizer**p[2])
C:\Users\johns\AppData\Local\Temp\ipykernel_7624\3385663312.py:8: RuntimeWarning: invalid value encountered in multiply
  return (px_given_y**p[0]) * (lift**p[1]) * (lift_normalizer**p[2])


5147.5
734.0
267.0
249.0
239.5
233.5
447.5
451.0
234.5
233.5
233.0
234.5
236.0
236.0
233.0
719.5
3077.0
624.5
423.5
241.5
436.5
226.0
269.5
229.5
224.5
224.0
271.0
223.5
272.0
223.5
223.5
223.5
541.5
390.5
198.5
233.0
222.0
213.0
195.0
192.5
186.5
215.0
188.5
184.5
829.0
184.5
170.0
271.5
213.5
141.0
124.0
156.5
127.5
115.5
162.0
119.0
163.0
117.0
163.5
117.0
115.5
474.5
1887.5
287.0
210.5
126.5
154.0
115.5
115.5
115.5
115.5
115.5
115.5
115.5
115.5
115.5
115.5
309.0
427.0
162.0
169.0
150.5
135.0
120.0
159.0
162.5
118.0
114.5
163.0
115.0
114.5
114.5
114.5
114.5
114.5
114.5
142.5
114.5
142.5
434.5
145.5
126.5
163.5
119.5
117.5
115.5
114.5
114.5
114.5
114.5
114.5
114.5
114.5
114.5
114.5


KeyboardInterrupt: 

In [ ]:
res

In [33]:
from tqdm.auto import tqdm

for ranking_type in tqdm(["px", "1/px", "px_given_y", "npmi_alt", "npmi", "npmi_fixed", "lift", "weighted_lift", "normalized_lift", "normalized_weighted_lift"]):
    metric = (np.argsort(-generate_ranking_score(a, ranking_type)).tolist().index(b) +
                np.argsort(-generate_ranking_score(b, ranking_type)).tolist().index(a))/2
    
    print(metric)

  0%|          | 0/10 [00:00<?, ?it/s]

5559.0
290301.0
495.0


C:\Users\johns\AppData\Local\Temp\ipykernel_13484\3915187622.py:15: RuntimeWarning: divide by zero encountered in log2
  return np.log2(lift) / np.log2(lift_normalizer)


402.5


C:\Users\johns\AppData\Local\Temp\ipykernel_13484\3915187622.py:17: RuntimeWarning: divide by zero encountered in log2
  return np.log2(lift) / -np.log2(px_given_y)
C:\Users\johns\AppData\Local\Temp\ipykernel_13484\3915187622.py:17: RuntimeWarning: divide by zero encountered in divide
  return np.log2(lift) / -np.log2(px_given_y)
C:\Users\johns\AppData\Local\Temp\ipykernel_13484\3915187622.py:17: RuntimeWarning: invalid value encountered in divide
  return np.log2(lift) / -np.log2(px_given_y)


63.5


C:\Users\johns\AppData\Local\Temp\ipykernel_13484\3915187622.py:19: RuntimeWarning: divide by zero encountered in log2
  return np.log2(lift) / -np.log2(pxy[a].toarray())
C:\Users\johns\AppData\Local\Temp\ipykernel_13484\3915187622.py:19: RuntimeWarning: invalid value encountered in divide
  return np.log2(lift) / -np.log2(pxy[a].toarray())


904.0
2132.0
127.5
187.5
284.0


In [28]:
pxy[a].toarray()

array([9.00e-06, 1.50e-05, 1.01e-04, ..., 0.00e+00, 0.00e+00, 0.00e+00],
      shape=(295860,))

In [236]:
from tqdm.auto import tqdm

for ranking_type in tqdm(["px", "px_given_y", "npmi_alt", "npmi", "lift", "weighted_lift", "normalized_lift", "normalized_weighted_lift"]):
    metric = (np.argsort(-generate_ranking_score(a, ranking_type)).tolist().index(b) +
                np.argsort(-generate_ranking_score(b, ranking_type)).tolist().index(a))/2
    
    print(metric)

  0%|          | 0/8 [00:00<?, ?it/s]

5559.0
495.0


C:\Users\johns\AppData\Local\Temp\ipykernel_17924\1428710982.py:15: RuntimeWarning: divide by zero encountered in log2
  return np.log2(lift) / np.log2(lift_normalizer)


402.5


C:\Users\johns\AppData\Local\Temp\ipykernel_17924\1428710982.py:17: RuntimeWarning: divide by zero encountered in log2
  return np.log2(lift) / -np.log2(px_given_y)
C:\Users\johns\AppData\Local\Temp\ipykernel_17924\1428710982.py:17: RuntimeWarning: divide by zero encountered in divide
  return np.log2(lift) / -np.log2(px_given_y)
C:\Users\johns\AppData\Local\Temp\ipykernel_17924\1428710982.py:17: RuntimeWarning: invalid value encountered in divide
  return np.log2(lift) / -np.log2(px_given_y)


63.5
2132.0
127.5
187.5
284.0


In [237]:
# score = generate_ranking_score(b, "normalized_weighted_lift")
# score = generate_ranking_score(b, "normalized_lift")
# score = generate_ranking_score(b, "weighted_lift")
score = generate_ranking_score(b, "npmi")
# score = generate_ranking_score(b, "npmi_alt")

print(np.argsort(-score).tolist().index(a))

13


C:\Users\johns\AppData\Local\Temp\ipykernel_17924\1428710982.py:17: RuntimeWarning: divide by zero encountered in log2
  return np.log2(lift) / -np.log2(px_given_y)
C:\Users\johns\AppData\Local\Temp\ipykernel_17924\1428710982.py:17: RuntimeWarning: divide by zero encountered in divide
  return np.log2(lift) / -np.log2(px_given_y)
C:\Users\johns\AppData\Local\Temp\ipykernel_17924\1428710982.py:17: RuntimeWarning: invalid value encountered in divide
  return np.log2(lift) / -np.log2(px_given_y)


In [238]:
top_k = 10

for i in np.argsort(-score)[:top_k]:
    print(idx2cat[i])

Hanzee (spotify:artist:5yM1po4NHvE2yE1Kf84LWJ)
Nils Bech (spotify:artist:57QhXfAsLsIRtgC1VfHu1F)
Rockettothesky (spotify:artist:0nu7qEOc8X8UFK10d8lsLw)
Elsa & Emilie (spotify:artist:4HDNQLqhooVfWXtIRMyqMY)
Ganic (spotify:artist:2qT4UNxM0aOz52gpSXuooT)
Feitn Fra Kolbotn (spotify:artist:5pTo5SvSAVsfJ9vbVwQyUt)
Gabrielle (spotify:artist:4OovmAu23KrDlDQI2UbneL)
ZL-Project (spotify:artist:4w8PGLhS3yzYSzeV3x2hkA)
Kjartan Lauritzen (spotify:artist:0TW5M8RYADmgeCP1q523hf)
Surferosa (spotify:artist:5WUYimkWakv41ZVVIq3BDr)


In [208]:
score = generate_ranking_score(a, "npmi")

np.argsort(-score).tolist().index(b)

C:\Users\johns\AppData\Local\Temp\ipykernel_17924\3250455684.py:13: RuntimeWarning: divide by zero encountered in log2
  return np.log2(lift) / -np.log2(px_given_y)
C:\Users\johns\AppData\Local\Temp\ipykernel_17924\3250455684.py:13: RuntimeWarning: divide by zero encountered in divide
  return np.log2(lift) / -np.log2(px_given_y)
C:\Users\johns\AppData\Local\Temp\ipykernel_17924\3250455684.py:13: RuntimeWarning: invalid value encountered in divide
  return np.log2(lift) / -np.log2(px_given_y)


114

In [211]:
score = generate_ranking_score(b, "npmi_alt")

np.argsort(-score).tolist().index(a)

C:\Users\johns\AppData\Local\Temp\ipykernel_17924\3250455684.py:11: RuntimeWarning: divide by zero encountered in log2
  return np.log2(lift) / np.log2(lift_normalizer)


753

In [212]:
top_k = 10

for i in np.argsort(-score)[:top_k]:
    print(idx2cat[i])

Highasakite (spotify:artist:5awQWdBpLqN2KFVRN8w56T)
Hanzee (spotify:artist:5yM1po4NHvE2yE1Kf84LWJ)
Nils Bech (spotify:artist:57QhXfAsLsIRtgC1VfHu1F)
Rockettothesky (spotify:artist:0nu7qEOc8X8UFK10d8lsLw)
Elsa & Emilie (spotify:artist:4HDNQLqhooVfWXtIRMyqMY)
Ganic (spotify:artist:2qT4UNxM0aOz52gpSXuooT)
Feitn Fra Kolbotn (spotify:artist:5pTo5SvSAVsfJ9vbVwQyUt)
Gabrielle (spotify:artist:4OovmAu23KrDlDQI2UbneL)
ZL-Project (spotify:artist:4w8PGLhS3yzYSzeV3x2hkA)
Kjartan Lauritzen (spotify:artist:0TW5M8RYADmgeCP1q523hf)


In [18]:
X.shape

(295860, 295860)

In [19]:
px = mat.mean(axis=0)

In [24]:
mat.shape

(1000000, 295860)

In [39]:
px_given_y = (mat[:, a:a+1] * mat).sum(axis=0) / mat[:, a].sum()

In [45]:
lift = px_given_y / px

In [48]:
top_k = 10

for i in np.argsort(-lift)[:top_k]:
    print(idx2cat[i])

Ásgeir (spotify:artist:7xUZ4069zcyBM4Bn10NQ1c)
Malachi Jackson (spotify:artist:6rfzWpkjEVWygReumyxJzc)
Jiggabits (spotify:artist:0Jw1CUzPv1WBB8uOUCj9yU)
Gilus (spotify:artist:50wkBXgHtMxRA905sYlUxP)
Ice Cream Cathedral (spotify:artist:4b75XvQ2tB53X7GoD5Cvhn)
New West Guitar Group (spotify:artist:6eGQni9pk1y5tsiZ5TBBID)
Jake Coronado (spotify:artist:5VW29S4YpWMDvohHbqUOLF)
Tuska (spotify:artist:3RjhhylIPk66I6HIuTUG5b)
Similia (spotify:artist:4TNlBKqH3ujY7r6fOrsJZx)
Robert Mitchell 3io (spotify:artist:05oAeVFNUPHOqBXDx19GNA)


In [49]:
weighted_lift = lift * px_given_y

In [50]:
top_k = 10

for i in np.argsort(-weighted_lift)[:top_k]:
    print(idx2cat[i])

Ásgeir (spotify:artist:7xUZ4069zcyBM4Bn10NQ1c)
James Vincent McMorrow (spotify:artist:7FDlvgcodNfC0IBdWevl4u)
José González (spotify:artist:6xrCU6zdcSTsG2hLrojpmI)
Sylvan Esso (spotify:artist:39vA9YljbnOApXKniLWBZv)
Ben Howard (spotify:artist:5schNIzWdI9gJ1QRK8SBnc)
Bon Iver (spotify:artist:4LEiUm1SRbFMgfqnQTwUbQ)
alt-J (spotify:artist:3XHO7cRUPCLOr6jwp8vsx5)
Dustin Tebbutt (spotify:artist:0z9hynUsIjf0ddI4uHqPWX)
Chet Faker (spotify:artist:2Q0MyH5YMI5HPQjFjlq5g3)
SOHN (spotify:artist:6XZYAWJLL8UIbxAqjKj3cg)


In [58]:
lift_normalizer = 1/np.where(px < px[a], px, px[a])

normalized_lift = lift / lift_normalizer

In [59]:
top_k = 10

for i in np.argsort(-normalized_lift)[:top_k]:
    print(idx2cat[i])

Ásgeir (spotify:artist:7xUZ4069zcyBM4Bn10NQ1c)
Dustin Tebbutt (spotify:artist:0z9hynUsIjf0ddI4uHqPWX)
Volcano Choir (spotify:artist:6gAtOqhriLzOzb3Qqmg5kO)
Lo-Fang (spotify:artist:5EDkJDlRNcMs3ewliB24QA)
Novo Amor (spotify:artist:0rZp7G3gIH6WkyeXbrZnGi)
PHOX (spotify:artist:3ix4iw2URncSdE7X292bXy)
SOHN (spotify:artist:6XZYAWJLL8UIbxAqjKj3cg)
The Kite String Tangle (spotify:artist:3D6cosC5ZOLCpRxt6T3XS7)
Nick Mulvey (spotify:artist:3x8FbPjh2Qz55XMdE2Yalj)
Vancouver Sleep Clinic (spotify:artist:77BznF1Dr1k5KyEZ6Nn3jB)


In [62]:
normalized_weighted_lift = normalized_lift * px_given_y

In [63]:
top_k = 10

for i in np.argsort(-normalized_weighted_lift)[:top_k]:
    print(idx2cat[i])

Ásgeir (spotify:artist:7xUZ4069zcyBM4Bn10NQ1c)
James Vincent McMorrow (spotify:artist:7FDlvgcodNfC0IBdWevl4u)
José González (spotify:artist:6xrCU6zdcSTsG2hLrojpmI)
Sylvan Esso (spotify:artist:39vA9YljbnOApXKniLWBZv)
Ben Howard (spotify:artist:5schNIzWdI9gJ1QRK8SBnc)
Bon Iver (spotify:artist:4LEiUm1SRbFMgfqnQTwUbQ)
alt-J (spotify:artist:3XHO7cRUPCLOr6jwp8vsx5)
Dustin Tebbutt (spotify:artist:0z9hynUsIjf0ddI4uHqPWX)
Chet Faker (spotify:artist:2Q0MyH5YMI5HPQjFjlq5g3)
SOHN (spotify:artist:6XZYAWJLL8UIbxAqjKj3cg)


In [188]:
# npmi_alt = lift / lift_normalizer
npmi_alt = np.log2(lift) / np.log2(lift_normalizer)

top_k = 10

for i in np.argsort(-npmi_alt)[:top_k]:
    print(idx2cat[i])

Ásgeir (spotify:artist:7xUZ4069zcyBM4Bn10NQ1c)
Dustin Tebbutt (spotify:artist:0z9hynUsIjf0ddI4uHqPWX)
Ásgeir Trausti (spotify:artist:7fNWySjsDn74LCawyJ27EQ)
Allman Brown (spotify:artist:239Y6QdFqVFfdsw6moqSEN)
Low Volts (spotify:artist:3PxUwSSsVaW0XyBiRJF2oS)
PHOX (spotify:artist:3ix4iw2URncSdE7X292bXy)
Volcano Choir (spotify:artist:6gAtOqhriLzOzb3Qqmg5kO)
Lo-Fang (spotify:artist:5EDkJDlRNcMs3ewliB24QA)
Nick Mulvey (spotify:artist:3x8FbPjh2Qz55XMdE2Yalj)
Novo Amor (spotify:artist:0rZp7G3gIH6WkyeXbrZnGi)


C:\Users\johns\AppData\Local\Temp\ipykernel_17924\3717324443.py:2: RuntimeWarning: divide by zero encountered in log2
  npmi_alt = np.log2(lift) / np.log2(lift_normalizer)


In [189]:
np.argsort(-npmi_alt).tolist().index(b)

52

In [175]:
npmi_alt = np.log2(lift) / np.log2(lift_normalizer)
# npmi_alt = np.log2(lift) / np.log2(1/lift_normalizer)

top_k = 10

for i in np.argsort(-npmi_alt)[:top_k]:
    print(idx2cat[i])

Ásgeir (spotify:artist:7xUZ4069zcyBM4Bn10NQ1c)
Dustin Tebbutt (spotify:artist:0z9hynUsIjf0ddI4uHqPWX)
Ásgeir Trausti (spotify:artist:7fNWySjsDn74LCawyJ27EQ)
Allman Brown (spotify:artist:239Y6QdFqVFfdsw6moqSEN)
Low Volts (spotify:artist:3PxUwSSsVaW0XyBiRJF2oS)
PHOX (spotify:artist:3ix4iw2URncSdE7X292bXy)
Volcano Choir (spotify:artist:6gAtOqhriLzOzb3Qqmg5kO)
Lo-Fang (spotify:artist:5EDkJDlRNcMs3ewliB24QA)
Nick Mulvey (spotify:artist:3x8FbPjh2Qz55XMdE2Yalj)
Novo Amor (spotify:artist:0rZp7G3gIH6WkyeXbrZnGi)


C:\Users\johns\AppData\Local\Temp\ipykernel_17924\2741643677.py:1: RuntimeWarning: divide by zero encountered in log2
  npmi_alt = np.log2(lift) / np.log2(lift_normalizer)


In [176]:
np.argsort(-npmi_alt).tolist().index(b)

52

In [135]:
npmi = np.log2(lift) / -np.log2(px_given_y)

top_k = 10

for i in np.argsort(-npmi)[:top_k]:
    print(idx2cat[i])

James Vincent McMorrow (spotify:artist:7FDlvgcodNfC0IBdWevl4u)
Ben Howard (spotify:artist:5schNIzWdI9gJ1QRK8SBnc)
alt-J (spotify:artist:3XHO7cRUPCLOr6jwp8vsx5)
Bon Iver (spotify:artist:4LEiUm1SRbFMgfqnQTwUbQ)
Sylvan Esso (spotify:artist:39vA9YljbnOApXKniLWBZv)
José González (spotify:artist:6xrCU6zdcSTsG2hLrojpmI)
Chet Faker (spotify:artist:2Q0MyH5YMI5HPQjFjlq5g3)
Hozier (spotify:artist:2FXC3k01G6Gw61bmprjgqS)
Glass Animals (spotify:artist:4yvcSjfu4PC0CYQyLy4wSq)
James Blake (spotify:artist:53KwLdlmrlCelAZMaLVZqU)


C:\Users\johns\AppData\Local\Temp\ipykernel_17924\876563672.py:1: RuntimeWarning: divide by zero encountered in log2
  npmi = np.log2(lift) / -np.log2(px_given_y)
C:\Users\johns\AppData\Local\Temp\ipykernel_17924\876563672.py:1: RuntimeWarning: divide by zero encountered in divide
  npmi = np.log2(lift) / -np.log2(px_given_y)
C:\Users\johns\AppData\Local\Temp\ipykernel_17924\876563672.py:1: RuntimeWarning: invalid value encountered in divide
  npmi = np.log2(lift) / -np.log2(px_given_y)


In [136]:
np.argsort(-npmi).tolist().index(b)

114

In [75]:
np.argsort(-lift).tolist().index(b)

2330

In [76]:
np.argsort(-weighted_lift).tolist().index(b)

134

In [77]:
np.argsort(-normalized_lift).tolist().index(b)

344

In [78]:
np.argsort(-normalized_weighted_lift).tolist().index(b)

566

In [80]:
from scipy.optimize import minimize_scalar

In [84]:
np.exp(np.log(px_given_y) * x) * normalized_lift

C:\Users\johns\AppData\Local\Temp\ipykernel_17924\1504623455.py:1: RuntimeWarning: divide by zero encountered in log
  np.exp(np.log(px_given_y) * x) * normalized_lift


NameError: name 'x' is not defined

In [ ]:
from scipy.optimize import minimize

In [125]:
def f(p):
#     ranking_metric = np.exp(np.log(px_given_y) * x) * normalized_lift
#     ranking_metric = np.exp(np.log(px_given_y) * x) * lift
#     ranking_metric = px_given_y**x * lift
#     ranking_metric = px_given_y**x * lift * (1/lift_normalizer)**y
    ranking_metric = np.exp(np.log(px) * p[0] + np.log(px_given_y) * p[1] + np.log(lift_normalizer) * p[2])
    
    return np.argsort(-ranking_metric).tolist().index(b)

# res = minimize_scalar(f)
res = minimize(f, [-1, 1, -1], method="Powell")
# res = minimize(f, [-1, 1, -1], method="Nelder-Mead")

res

C:\Users\johns\AppData\Local\Temp\ipykernel_17924\3828707080.py:6: RuntimeWarning: divide by zero encountered in log
  ranking_metric = np.exp(np.log(px) * p[0] + np.log(px_given_y) * p[1] + np.log(lift_normalizer) * p[2])


 message: Optimization terminated successfully.
 success: True
  status: 0
     fun: 36
       x: [-1.577e+00  9.978e-01 -1.080e+00]
     nit: 4
   direc: [[ 1.000e+00  0.000e+00  0.000e+00]
           [ 0.000e+00  1.000e+00  0.000e+00]
           [ 0.000e+00  0.000e+00  1.000e+00]]
    nfev: 312

In [123]:
res.x

array([-1.57702903,  0.99782948, -1.07984841])

In [121]:
f([-1, 1, -1])

C:\Users\johns\AppData\Local\Temp\ipykernel_17924\3487729486.py:6: RuntimeWarning: divide by zero encountered in log
  ranking_metric = np.exp(np.log(px) * p[0] + np.log(px_given_y) * p[1] * np.log(lift_normalizer) * p[2])


293862

In [97]:
res.x

array([-0.99476447,  0.99770523])

In [109]:
1/(px_given_y**0.99476447)

C:\Users\johns\AppData\Local\Temp\ipykernel_17924\3665458016.py:1: RuntimeWarning: divide by zero encountered in divide
  1/(px_given_y**0.99476447)


array([213.98187753, 128.73295547,  19.3106036 , ...,          inf,
                inf,          inf], shape=(295860,))

In [105]:
px_given_y**-0.99476447

C:\Users\johns\AppData\Local\Temp\ipykernel_17924\3173790207.py:1: RuntimeWarning: divide by zero encountered in power
  px_given_y**-0.99476447


array([213.98187753, 128.73295547,  19.3106036 , ...,          inf,
                inf,          inf], shape=(295860,))

In [111]:
# ranking_metric = lift * (px_given_y) * (1/lift_normalizer)
ranking_metric = 1/(px_given_y**0.99476447) * lift * (1/lift_normalizer)**0.99770523

np.argsort(-ranking_metric).tolist().index(b)

C:\Users\johns\AppData\Local\Temp\ipykernel_17924\3998867154.py:2: RuntimeWarning: divide by zero encountered in divide
  ranking_metric = 1/(px_given_y**0.99476447) * lift * (1/lift_normalizer)**0.99770523
C:\Users\johns\AppData\Local\Temp\ipykernel_17924\3998867154.py:2: RuntimeWarning: invalid value encountered in multiply
  ranking_metric = 1/(px_given_y**0.99476447) * lift * (1/lift_normalizer)**0.99770523


27

In [70]:
# import pandas as pd

# pd.DataFrame({
#     "a": normalized_weighted_lift,
#     "b": weighted_lift
# }).corr("kendall")["a"]["b"]

In [72]:
# import pandas as pd

# pd.DataFrame({
#     "a": normalized_lift,
#     "b": lift
# }).corr("kendall")["a"]["b"]

In [74]:
# import pandas as pd

# pd.DataFrame({
#     "a": weighted_lift,
#     "b": lift
# }).corr("kendall")["a"]["b"]